# Paper 02: Exploratory Analysis
Temporal trends, spatial patterns, dose-response relationships, and regional comparisons
for heat stress variables and cattle slaughter (1984-2025).

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

project_root = Path('/Users/klesinger/Library/CloudStorage/GoogleDrive-kdl0040@uah.edu/My Drive/VEDA/Stories/livestock_and_heat/research')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from src.analysis import mann_kendall_test

plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})
sns.set_style('whitegrid')

df = pd.read_csv(config.PAPER_ANALYSIS_FILE, parse_dates=['week_ending'])
config.PAPER_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loaded: {df.shape}, {df['year'].min()}-{df['year'].max()}")

## 1. Per-Variable Trend Analysis (Mann-Kendall)

Long-term trends for every climate variable across the full 41-year record.

**Why annual means (not weekly)?** The Mann-Kendall test assumes independent observations.
Weekly data has strong serial autocorrelation — consecutive weeks within a season are
highly correlated, and the dominant seasonal cycle would cause nearly every variable to
appear "significant" even without any long-term trend. Aggregating to **annual means**
removes seasonality and isolates the true multi-decadal climate signal. This is standard
practice in climate trend literature (e.g., IPCC AR6 methodology, Wilks 2011 *Statistical
Methods in the Atmospheric Sciences*).

A **seasonal Mann-Kendall** analysis follows in Section 1b, testing trends within each
season separately — e.g., whether summers are getting hotter faster than the annual
average suggests.

In [ ]:
# Compute annual means per variable per region
climate_vars = [c for c in df.columns if c.startswith('mean_') and '_lag' not in c and '_roll' not in c and '_has' not in c and '_log1p' not in c]

trend_results = []

for region in df['region'].unique():
    rdata = df[df['region'] == region]
    annual = rdata.groupby('year')[climate_vars].mean()
    
    for var in climate_vars:
        series = annual[var].dropna().values
        if len(series) >= 10:
            mk = mann_kendall_test(series)
            trend_results.append({
                'region': region,
                'variable': var,
                'z_stat': mk['z'],
                'p_value': mk['p'],
                'trend': mk['trend'],
                'significant': mk['p'] < 0.05,
                'annual_mean': annual[var].mean(),
                'first_decade_mean': annual[var].iloc[:10].mean(),
                'last_decade_mean': annual[var].iloc[-10:].mean(),
            })

trend_df = pd.DataFrame(trend_results)

print("=== Significant Trends (p < 0.05) ===\n")
sig = trend_df[trend_df['significant']].sort_values('p_value')
for _, row in sig.iterrows():
    change = row['last_decade_mean'] - row['first_decade_mean']
    print(f"{row['region']:>10s} | {row['variable']:<45s} | {row['trend']:>10s} | z={row['z_stat']:+.2f} | p={row['p_value']:.4f} | Change: {change:+.2f}")

print(f"\n{len(sig)} / {len(trend_df)} variable-region combinations show significant trends")

In [ ]:
# Plot annual means for the most important variables
key_vars = ['mean_vpd_mean', 'mean_vpd_max', 'mean_daytime_hours_above_30',
            'mean_daytime_hours_above_35', 'mean_nighttime_hours_above_21',
            'mean_nighttime_hours_above_24']

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)

for ax, var in zip(axes.flat, key_vars):
    for region in df['region'].unique():
        rdata = df[df['region'] == region]
        annual = rdata.groupby('year')[var].mean()
        label = region.replace('region_', 'Region ')
        ax.plot(annual.index, annual.values, label=label, linewidth=1.5)
        
        x = np.arange(len(annual))
        slope, intercept, _, _, _ = stats.linregress(x, annual.values)
        ax.plot(annual.index, intercept + slope * x, '--', alpha=0.5)
    
    mk_info = trend_df[(trend_df['variable'] == var)].iloc[0]
    sig_marker = '*' if mk_info['significant'] else ''
    short_name = var.replace('mean_', '').replace('_', ' ').title()
    ax.set_title(f"{short_name}{sig_marker}")
    ax.legend(fontsize=8)

axes[-1, 0].set_xlabel('Year')
axes[-1, 1].set_xlabel('Year')
plt.suptitle('Annual Mean Climate Variables (1984-2025)\n* = significant Mann-Kendall trend', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig02_variable_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 1b. Seasonal Mann-Kendall Analysis

The annual mean trend can mask important seasonal differences. For example, summer
nighttime temperatures may be rising faster than the annual average suggests, which
is critical for cattle that rely on nighttime cooling for heat dissipation.

Here we compute Mann-Kendall trends on **seasonal means** — one value per year per
season (e.g., summer 1984 mean, summer 1985 mean, ..., summer 2025 mean). This gives
41 independent data points per season, preserving the non-parametric assumption while
revealing whether the trend is concentrated in the heat-stress-relevant months.

In [ ]:
# Seasonal Mann-Kendall: test trends within each season independently
seasons = config.SEASONS
focus_vars = ['mean_vpd_mean', 'mean_daytime_hours_above_30', 'mean_daytime_hours_above_35',
              'mean_nighttime_hours_above_21', 'mean_nighttime_hours_above_24']

seasonal_trends = []

for region in df['region'].unique():
    rdata = df[df['region'] == region]
    for season_name, months in seasons.items():
        season_data = rdata[rdata['month'].isin(months)]
        seasonal_annual = season_data.groupby('year')[focus_vars].mean()
        
        for var in focus_vars:
            series = seasonal_annual[var].dropna().values
            if len(series) >= 10:
                mk = mann_kendall_test(series)
                seasonal_trends.append({
                    'region': region,
                    'season': season_name,
                    'variable': var,
                    'z_stat': mk['z'],
                    'p_value': mk['p'],
                    'trend': mk['trend'],
                    'significant': mk['p'] < 0.05,
                    'first_decade': seasonal_annual[var].iloc[:10].mean(),
                    'last_decade': seasonal_annual[var].iloc[-10:].mean(),
                })

seasonal_trend_df = pd.DataFrame(seasonal_trends)

print("=== Seasonal vs Annual Trends (focus: heat stress variables) ===\n")
print(f"{'Region':<12s} {'Season':<8s} {'Variable':<40s} {'z-stat':>8s} {'p-value':>10s} {'Sig':>5s} {'Change':>8s}")
print("-" * 95)

for _, row in seasonal_trend_df.sort_values(['variable', 'region', 'season']).iterrows():
    change = row['last_decade'] - row['first_decade']
    sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else ''
    print(f"{row['region']:<12s} {row['season']:<8s} {row['variable']:<40s} {row['z_stat']:>+8.2f} {row['p_value']:>10.4f} {sig:>5s} {change:>+8.2f}")

print("\n=== Strongest Seasonal Trend per Variable ===\n")
for var in focus_vars:
    for region in df['region'].unique():
        subset = seasonal_trend_df[(seasonal_trend_df['variable'] == var) & (seasonal_trend_df['region'] == region)]
        strongest = subset.loc[subset['z_stat'].abs().idxmax()]
        short = var.replace('mean_', '')
        print(f"  {region} {short}: strongest in {strongest['season']} (z={strongest['z_stat']:+.2f}, p={strongest['p_value']:.4f})")

## 2. Seasonal Patterns

Monthly distribution of heat stress variables showing the annual cycle.

These boxplots reveal which months drive the heat stress signal. For livestock modeling,
the key insight is the **asymmetry** of the annual cycle — heat stress hours ramp up
sharply in May-June but persist through September, while VPD peaks in July-August.
This asymmetry matters because cattle acclimation lags behind temperature changes
by 9-14 days, making early-season heat events disproportionately dangerous.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, var in zip(axes.flat, key_vars):
    plot_data = df[['month', var, 'region']].copy()
    sns.boxplot(data=plot_data, x='month', y=var, hue='region', ax=ax, 
                fliersize=1, linewidth=0.8)
    short_name = var.replace('mean_', '').replace('_', ' ').title()
    ax.set_title(short_name)
    ax.set_xlabel('Month')
    ax.legend(fontsize=7)

plt.suptitle('Monthly Distribution of Climate Variables by Region', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig03_seasonal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. VPD Dose-Response Curves

Relationship between VPD and cattle slaughter — the dominant predictor from our
feature importance analysis.

**Why VPD over temperature?** VPD measures the atmospheric "drying power" — the
difference between how much moisture the air *can* hold and how much it *does* hold.
High VPD impairs evaporative cooling (sweating/panting), the primary thermoregulation
mechanism for cattle. Unlike raw temperature, VPD captures humidity effects: a 35C day
at 20% humidity (VPD ~3.5 kPa) is far less lethal than 35C at 60% humidity (VPD ~1.5 kPa).

The quadratic fit tests for a non-linear dose-response — whether mortality accelerates
above a critical VPD threshold rather than increasing linearly. Error bars show standard
error of the mean within each VPD bin (minimum 5 observations per bin).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, region in zip(axes, df['region'].unique()):
    rdata = df[df['region'] == region].copy()
    rdata['vpd_bin'] = pd.cut(rdata['mean_vpd_mean'], bins=20)
    binned = rdata.groupby('vpd_bin', observed=True)['slaughter_beef_dairy'].agg(['mean', 'std', 'count'])
    binned['midpoint'] = [interval.mid for interval in binned.index]
    binned = binned[binned['count'] >= 5]
    
    ax.errorbar(binned['midpoint'], binned['mean'], yerr=binned['std']/np.sqrt(binned['count']),
                fmt='o-', capsize=3, markersize=5)
    
    x = binned['midpoint'].values
    y = binned['mean'].values
    if len(x) > 3:
        coeffs = np.polyfit(x, y, 2)
        x_smooth = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_smooth, np.polyval(coeffs, x_smooth), 'r--', label='Quadratic fit')
    
    label = region.replace('region_', 'Region ')
    ax.set_title(f'{label}: VPD vs Cattle Slaughter')
    ax.set_xlabel('Weekly Mean VPD (kPa)')
    ax.set_ylabel('Slaughter (1000 head/wk)')
    ax.legend()

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig04_vpd_dose_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Nighttime Recovery Analysis

Hours above 24C at night vs cattle mortality — the recovery failure mechanism.

**Physiological basis:** Cattle dissipate ~80% of accumulated daytime heat load during
nighttime hours through radiant and convective cooling. When nighttime temperatures
remain above ~24C, this recovery is impaired and heat accumulates across consecutive
days (Hahn 1999, Mader et al. 2006). The 24C threshold corresponds to the point where
cattle can no longer maintain thermal equilibrium through passive cooling alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, region in zip(axes, df['region'].unique()):
    rdata = df[df['region'] == region].copy()
    rdata['night_bin'] = pd.cut(rdata['mean_nighttime_hours_above_24'], bins=15)
    binned = rdata.groupby('night_bin', observed=True)['slaughter_beef_dairy'].agg(['mean', 'std', 'count'])
    binned['midpoint'] = [interval.mid for interval in binned.index]
    binned = binned[binned['count'] >= 5]
    
    ax.errorbar(binned['midpoint'], binned['mean'], yerr=binned['std']/np.sqrt(binned['count']),
                fmt='s-', capsize=3, markersize=5, color='darkred')
    
    label = region.replace('region_', 'Region ')
    ax.set_title(f'{label}: Nighttime Heat (>24C) vs Slaughter')
    ax.set_xlabel('Weekly Nighttime Hours Above 24C')
    ax.set_ylabel('Slaughter (1000 head/wk)')

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig05_nighttime_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Compound Stress: VPD x Temperature Interaction

Heat stress mortality is not simply additive — high VPD combined with high temperature
produces a **synergistic** effect greater than the sum of individual stressors. This is
because VPD impairs evaporative cooling precisely when the animal needs it most (during
extreme heat).

This scatter plot uses summer-only data (June-August) to isolate the compound stress
signal. Color intensity represents slaughter volume.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, region in zip(axes, df['region'].unique()):
    rdata = df[df['region'] == region].copy()
    summer = rdata[rdata['month'].isin([6, 7, 8])]
    
    if len(summer) > 50:
        sc = ax.scatter(summer['mean_daytime_hours_above_30'], summer['mean_vpd_mean'],
                       c=summer['slaughter_beef_dairy'], cmap='YlOrRd', s=15, alpha=0.6)
        plt.colorbar(sc, ax=ax, label='Slaughter (1000 hd/wk)')
    
    label = region.replace('region_', 'Region ')
    ax.set_title(f'{label}: Summer Compound Stress')
    ax.set_xlabel('Daytime Hours Above 30C')
    ax.set_ylabel('Weekly Mean VPD (kPa)')

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig06_compound_stress.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cumulative Exposure Windows

Compare correlation strength at different rolling window sizes (current week vs 2/4/8-week
rolling averages).

**Rationale:** Heat stress effects are cumulative — a single hot week may not kill cattle,
but sustained exposure over multiple weeks depletes physiological reserves and overwhelms
adaptive capacity. The optimal window size tells us the "thermal memory" of the
cattle-mortality system.

In [ ]:
window_vars = {}
for var_base in ['mean_vpd_mean', 'mean_daytime_hours_above_35', 'mean_nighttime_hours_above_24']:
    window_vars[var_base] = {'current': var_base}
    for w in [2, 4, 8]:
        roll_col = f'{var_base}_roll{w}'
        if roll_col in df.columns:
            window_vars[var_base][f'{w}-week'] = roll_col

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (var_base, windows) in zip(axes, window_vars.items()):
    for region in df['region'].unique():
        rdata = df[df['region'] == region]
        corrs = []
        labels = []
        for wname, wcol in windows.items():
            if wcol in rdata.columns:
                r, _ = stats.pearsonr(rdata[wcol].dropna(), rdata.loc[rdata[wcol].notna(), 'slaughter_beef_dairy'])
                corrs.append(abs(r))
                labels.append(wname)
        label = region.replace('region_', 'R')
        ax.plot(labels, corrs, 'o-', label=label)
    
    short = var_base.replace('mean_', '').replace('_', ' ').title()
    ax.set_title(f'{short}')
    ax.set_ylabel('|Pearson r|')
    ax.legend()
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Correlation Strength vs Cumulative Window Size', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig07_window_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Regional Comparison: Region 4 (SE) vs Region 6 (South Central)

USDA cattle slaughter Region 4 (AL, FL, GA, KY, MS, NC, SC, TN) and Region 6 (AR, LA,
NM, OK, TX) represent distinct climate-livestock systems:
- **Region 4 (Southeast):** Humid subtropical — high humidity amplifies heat stress
  through VPD, but absolute temperatures are moderate
- **Region 6 (South Central):** Semi-arid to subtropical — Texas dominates with extreme
  temperatures but lower humidity; higher baseline slaughter volume

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for region in df['region'].unique():
    rdata = df[df['region'] == region]
    annual = rdata.groupby('year')['slaughter_beef_dairy'].mean()
    label = region.replace('region_', 'Region ')
    ax.plot(annual.index, annual.values, label=label, linewidth=2)
ax.set_title('Annual Mean Slaughter')
ax.set_ylabel('1000 head/week')
ax.legend()

ax = axes[0, 1]
for region in df['region'].unique():
    rdata = df[df['region'] == region]
    annual = rdata.groupby('year')['mean_vpd_mean'].mean()
    label = region.replace('region_', 'Region ')
    ax.plot(annual.index, annual.values, label=label, linewidth=2)
ax.set_title('Annual Mean VPD')
ax.set_ylabel('kPa')
ax.legend()

ax = axes[1, 0]
for region in df['region'].unique():
    rdata = df[df['region'] == region]
    summer = rdata[rdata['month'].isin([6, 7, 8])]
    label = region.replace('region_', 'Region ')
    ax.hist(summer['mean_daytime_hours_above_35'], bins=30, alpha=0.5, label=label, density=True)
ax.set_title('Summer Hours Above 35C (Distribution)')
ax.set_xlabel('Hours/week')
ax.legend()

ax = axes[1, 1]
key_corr_vars = ['slaughter_beef_dairy', 'mean_vpd_mean', 'mean_vpd_max',
                 'mean_daytime_hours_above_30', 'mean_daytime_hours_above_35',
                 'mean_nighttime_hours_above_21', 'mean_nighttime_hours_above_24']
existing = [c for c in key_corr_vars if c in df.columns]
corr_mat = df[existing].corr()
short_labels = [c.replace('mean_', '').replace('slaughter_', 'slaughter\n').replace('_', ' ')[:20] for c in existing]
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=short_labels, yticklabels=short_labels, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig08_regional_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Lag Analysis

Correlation between lagged climate variables (1-8 weeks prior) and current-week slaughter.

**Why lags matter:** Heat-induced cattle mortality is rarely instantaneous. Physiological
deterioration, decision-making by producers (to send stressed animals to slaughter early),
and supply chain delays all introduce temporal lags between heat exposure and observed
slaughter counts. The peak lag identifies the dominant response timescale.

In [ ]:
lag_vars_base = ['mean_vpd_mean', 'mean_daytime_hours_above_35', 'mean_nighttime_hours_above_24']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var_base in zip(axes, lag_vars_base):
    for region in df['region'].unique():
        rdata = df[df['region'] == region]
        lags = [0, 1, 2, 3, 4, 8]
        corrs = []
        for lag in lags:
            col = var_base if lag == 0 else f'{var_base}_lag{lag}'
            if col in rdata.columns:
                valid = rdata[[col, 'slaughter_beef_dairy']].dropna()
                r, _ = stats.pearsonr(valid[col], valid['slaughter_beef_dairy'])
                corrs.append(r)
            else:
                corrs.append(np.nan)
        label = region.replace('region_', 'R')
        ax.plot(lags, corrs, 'o-', label=label)
    
    short = var_base.replace('mean_', '').replace('_', ' ').title()
    ax.set_title(f'{short}')
    ax.set_xlabel('Lag (weeks)')
    ax.set_ylabel('Pearson r')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.legend()

plt.suptitle('Lag Correlation: Climate Variables vs Slaughter', fontsize=13)
plt.tight_layout()
plt.savefig(config.PAPER_FIGURES_DIR / 'fig09_lag_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary Statistics Table (for paper)

Descriptive statistics for the manuscript's Table 1. Reports N, mean, standard deviation,
min, 5th/50th/95th percentiles, and max for all key variables, stratified by region.
Also exports Mann-Kendall trend results (annual and seasonal) as supplementary tables.

In [ ]:
target = 'slaughter_beef_dairy'
summary_vars = [target] + key_vars

summary_rows = []
for var in summary_vars:
    if var not in df.columns:
        continue
    for region in df['region'].unique():
        rdata = df[df['region'] == region][var].dropna()
        summary_rows.append({
            'Variable': var.replace('mean_', ''),
            'Region': region.replace('region_', 'R'),
            'N': len(rdata),
            'Mean': rdata.mean(),
            'Std': rdata.std(),
            'Min': rdata.min(),
            'P5': rdata.quantile(0.05),
            'Median': rdata.median(),
            'P95': rdata.quantile(0.95),
            'Max': rdata.max(),
        })

summary_df = pd.DataFrame(summary_rows)
print("=== Summary Statistics (Table 1 for Paper) ===\n")
print(summary_df.to_string(index=False, float_format='%.2f'))

summary_df.to_csv(config.PAPER_FIGURES_DIR / 'table1_summary_stats.csv', index=False)
print(f"\nSaved: {config.PAPER_FIGURES_DIR / 'table1_summary_stats.csv'}")

trend_df.to_csv(config.PAPER_FIGURES_DIR / 'table2_trend_results.csv', index=False)
print(f"Saved: {config.PAPER_FIGURES_DIR / 'table2_trend_results.csv'}")

seasonal_trend_df.to_csv(config.PAPER_FIGURES_DIR / 'table3_seasonal_trends.csv', index=False)
print(f"Saved: {config.PAPER_FIGURES_DIR / 'table3_seasonal_trends.csv'}")